# AIMO 3 Training: SFT then GRPO (Lambda Labs Edition)

# Two-stage training pipeline:
#   Stage 1: SFT (Supervised Fine-Tuning)    = "Learn from examples"
#   Stage 2: GRPO (Group RL Policy Opt)       = "Learn from trying"
#
# Runtime: ~5-6 hours total on Lambda A100 40GB
# Cost: ~$9-10
# After training: upload to HuggingFace → attach to Kaggle submission notebook

---
## Part 0: Setup

We use **Unsloth** - a library that makes fine-tuning 2-5x faster and uses 70% less memory.
It achieves this by fusing GPU operations and optimizing memory management.

In [ ]:
%%capture
# Lambda Labs has a clean environment - install everything properly
!pip install unsloth
!pip install trl==0.18.1
!pip install vllm  # Installs cleanly on Lambda (unlike Kaggle)
!pip install huggingface_hub  # For uploading the final model

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
# You should see: GPU: NVIDIA H100, VRAM: ~80 GB

---
## Part 1: Understanding the Training Data

We use **NuminaMath-TIR** (Tool-Integrated Reasoning) - 72K math problems where each
solution uses Python code to compute intermediate steps.

This is the same dataset that won AIMO 1. The format looks like:

```
User: Find the sum of all primes less than 10.

Assistant: I need to find all prime numbers less than 10 and sum them.
Let me write code to do this.

```python
primes = [p for p in range(2, 10) if all(p % i != 0 for i in range(2, p))]
print(sum(primes))
```

```output
17
```

The prime numbers less than 10 are 2, 3, 5, 7.
Their sum is 2 + 3 + 5 + 7 = \boxed{17}
```

**Why TIR?** Models that use code as a calculator are MUCH more accurate than
models that try to do all arithmetic in their head. The code acts as a "tool"
that eliminates computation errors.

In [ ]:
from datasets import load_dataset

# Load the TIR dataset (72K problems with code-augmented solutions)
ds_tir = load_dataset("AI-MO/NuminaMath-TIR", split="train")
print(f"Training examples: {len(ds_tir)}")
print(f"Columns: {ds_tir.column_names}")

In [ ]:
# Let's look at one example to understand the format
sample = ds_tir[0]

print("=== PROBLEM ===")
print(sample["problem"][:300])
print()
print("=== MESSAGES FORMAT ===")
# The 'messages' field is already in chat format - this is what we train on
for msg in sample["messages"]:
    role = msg["role"]
    content = msg["content"][:200] + "..." if len(msg["content"]) > 200 else msg["content"]
    print(f"\n[{role.upper()}]")
    print(content)

---
## Part 2: SFT (Supervised Fine-Tuning)

### What is SFT?

SFT is the simplest form of training: **show the model examples, make it mimic them.**

```
Input:  (problem, correct_solution) pairs
Loss:   Cross-entropy on the solution tokens
Result: Model learns the FORMAT - how to write step-by-step, when to use code,
        how to format \boxed{answer}
```

Think of it like a student copying worked examples from a textbook. They learn
the *pattern* of problem-solving, but haven't truly internalized *reasoning*.

### What is LoRA?

A 7B model has 7 billion parameters. Training ALL of them needs ~56GB of memory
(weights + gradients + optimizer states). That won't fit on most GPUs.

**LoRA (Low-Rank Adaptation)** is a trick: instead of updating all 7B parameters,
we freeze the original weights and add tiny trainable matrices ("adapters") to each layer.

```
Original:  Y = W * X           (W is frozen, 7B params)
With LoRA: Y = W * X + B * A * X  (A and B are tiny, ~50M params)
```

- **r (rank)**: Size of A and B matrices. Higher = more capacity, more memory.
  r=64 is a good balance.
- **lora_alpha**: Scaling factor. Usually 16 or 2*r.
- Memory drops from ~56GB to ~16GB. Fits easily on H100!

In [ ]:
from unsloth import FastLanguageModel

# Load the base model in 4-bit quantization
# WHY Qwen2.5-Math-7B? It's already pre-trained on math data,
# so SFT on TIR teaches it the code-execution format on top of
# its existing math knowledge. Starting from a math model = faster convergence.

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Qwen/Qwen2.5-Math-7B-Instruct",
    max_seq_length=2048,    # Max tokens per training example
    load_in_4bit=True,      # 4-bit quantization: 7B model uses only ~4GB VRAM
    dtype=None,             # Auto-detect (bf16 on H100)
)

print(f"Model loaded! Parameters: {model.num_parameters():,}")

In [ ]:
from trl import SFTTrainer, SFTConfig

# Configure SFT training
# KEY HYPERPARAMETERS explained:

sft_config = SFTConfig(
    output_dir="./sft_checkpoint",
    
    # --- Batch size ---
    # effective_batch = per_device * gradient_accumulation = 2 * 8 = 16
    # Larger batch = more stable gradients but slower per-step
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    
    # --- Learning rate ---
    # With LoRA, you can use higher LR than full fine-tuning (2e-4 vs 2e-5)
    # because you're updating fewer parameters
    learning_rate=2e-4,
    lr_scheduler_type="cosine",  # Gradually reduce LR (like simulated annealing)
    warmup_steps=50,             # Slowly increase LR at start to avoid instability
    
    # --- Duration ---
    # 1 epoch = see every example once. 2 epochs is enough for SFT.
    # More epochs risk overfitting (memorizing instead of learning).
    num_train_epochs=2,
    
    # --- Precision ---
    # bf16 (bfloat16) is preferred on H100 - same speed as fp16 but no overflow issues
    bf16=True,
    fp16=False,
    
    # --- Logging & saving ---
    logging_steps=25,
    save_steps=1000,
    save_total_limit=2,  # Keep only last 2 checkpoints (save disk space)
    
    # --- Data ---
    max_seq_length=2048,
    dataset_text_field="text",  # Column name in our formatted dataset
    packing=False,        # Disabled - Unsloth/TRL version mismatch on this Kaggle image
    padding_free=False,   # MUST be False - Unsloth auto-enables this and it causes
                          # shape mismatch in fused cross_entropy loss (input 2048 vs target 3503)
    
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds_formatted,
    args=sft_config,
)

print(f"Total training steps: ~{len(ds_formatted) * 2 // 16}")
print(f"Estimated time on H100: ~3-4 hours")

In [ ]:
# Format the training data
# The NuminaMath-TIR dataset has a 'messages' field in chat format.
# We need to convert it to the model's specific chat template.
#
# Every model has its own template, e.g.:
#   Qwen:  <|im_start|>user\nProblem<|im_end|>\n<|im_start|>assistant\nSolution...
#   Llama: [INST] Problem [/INST] Solution...
#
# The tokenizer.apply_chat_template() handles this automatically.

def format_for_sft(example):
    """Convert messages to the model's chat format string."""
    messages = example["messages"]
    # apply_chat_template converts the message list to the model's expected format
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,              # Return string, not token IDs
        add_generation_prompt=False,  # Don't add trailing prompt
    )
    return {"text": text}

# Apply formatting to all examples
ds_formatted = ds_tir.map(format_for_sft, num_proc=4)

# Let's see what one formatted example looks like
print(ds_formatted[0]["text"][:500])
print("...")

In [ ]:
from trl import SFTTrainer, SFTConfig

# Configure SFT training
# KEY HYPERPARAMETERS explained:

sft_config = SFTConfig(
    output_dir="./sft_checkpoint",
    
    # --- Batch size ---
    # effective_batch = per_device * gradient_accumulation = 2 * 8 = 16
    # Larger batch = more stable gradients but slower per-step
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    
    # --- Learning rate ---
    # With LoRA, you can use higher LR than full fine-tuning (2e-4 vs 2e-5)
    # because you're updating fewer parameters
    learning_rate=2e-4,
    lr_scheduler_type="cosine",  # Gradually reduce LR (like simulated annealing)
    warmup_steps=50,             # Slowly increase LR at start to avoid instability
    
    # --- Duration ---
    # 1 epoch = see every example once. 2 epochs is enough for SFT.
    # More epochs risk overfitting (memorizing instead of learning).
    num_train_epochs=2,
    
    # --- Precision ---
    # bf16 (bfloat16) is preferred on H100 - same speed as fp16 but no overflow issues
    bf16=True,
    fp16=False,
    
    # --- Logging & saving ---
    logging_steps=25,
    save_steps=1000,
    save_total_limit=2,  # Keep only last 2 checkpoints (save disk space)
    
    # --- Data ---
    max_seq_length=2048,
    dataset_text_field="text",  # Column name in our formatted dataset
    packing= False,  # Pack multiple short examples into one sequence (faster!)
    padding_free=False,   # MUST be False - Unsloth auto-enables this and it causes 42                           
                         # shape mismatch in fused cross_entropy loss (iput 2048 vs target 3503)
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds_formatted,
    args=sft_config,
)

# Show estimated training time
steps = trainer.state.max_steps if hasattr(trainer.state, 'max_steps') else len(trainer.get_train_dataloader()) * sft_config.num_train_epochs
print(f"Total training steps: ~{len(ds_formatted) * 2 // 16}")
print(f"Estimated time on H100: ~2-3 hours")

In [ ]:
# ============================================
# RUN SFT TRAINING
# ============================================
# This will take ~2-3 hours on H100.
# Watch the loss - it should decrease from ~2.0 to ~0.5-0.8
#
# What's happening under the hood:
# 1. Feed a (problem, solution) pair to the model
# 2. Model predicts next token at each position
# 3. Compare prediction to actual next token (cross-entropy loss)
# 4. Backpropagate gradients through LoRA adapters only
# 5. Update adapter weights with AdamW optimizer
# 6. Repeat for every example in the dataset

trainer_stats = trainer.train()
print(f"\nTraining complete!")
print(f"Final loss: {trainer_stats.training_loss:.4f}")

In [ ]:
# Save the SFT checkpoint
# This saves ONLY the LoRA adapters (~200MB), not the full model (~14GB)
model.save_pretrained("./sft_checkpoint")
tokenizer.save_pretrained("./sft_checkpoint")
print("SFT checkpoint saved!")

---
## Resuming: Load from SFT Checkpoint

**If your kernel restarted** or you're starting a new session, run the cells below
to reload the model from your SFT checkpoint and jump straight to GRPO.

**Cells to run in order:**
1. Cell 2 (install)
2. Cell 3 (GPU check)
3. The "Load from SFT checkpoint" cell below
4. Cell 16 (reward function)
5. Cells 17-18 (GRPO data)
6. Cell 19 (GRPO setup — has resume support built in)
7. Cell 20 (train)

**Where are my checkpoints?**
- SFT: `./sft_checkpoint/` (LoRA adapters, ~200MB)
- GRPO: `./grpo_checkpoint/checkpoint-{step}/` (auto-saved every 50 steps)

On Kaggle, `/kaggle/working/` persists within a session. If your session dies,
save checkpoints as a Kaggle Dataset (Output tab) and re-attach next time.

In [ ]:
# ============================================
# LOAD MODEL FROM SFT CHECKPOINT (skip SFT training)
# ============================================
# Run this instead of cells 8-13 if you already have an SFT checkpoint.

from unsloth import FastLanguageModel
import os

# Where is your SFT checkpoint?
# Option A: Same Kaggle session (saved to /kaggle/working/)
SFT_CHECKPOINT = "./sft_checkpoint"
# Option B: Attached as Kaggle dataset input (uncomment and adjust):
# SFT_CHECKPOINT = "/kaggle/input/YOUR-DATASET-NAME/sft_checkpoint"

if not os.path.exists(SFT_CHECKPOINT):
    raise FileNotFoundError(
        f"SFT checkpoint not found at {SFT_CHECKPOINT}\n"
        "If you're in a new session, attach your checkpoint as a Kaggle Dataset input."
    )

# Load base model + apply saved LoRA adapters in one step
# Unsloth's from_pretrained detects LoRA adapters automatically
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_CHECKPOINT,
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)

print(f"✓ Model loaded from SFT checkpoint: {SFT_CHECKPOINT}")
print(f"  Parameters: {model.num_parameters():,}")
print(f"  Now skip to the GRPO cells (reward function → data → training)")

In [ ]:
# Format problems as prompts for GRPO
from datasets import Features, Value, Sequence

def make_grpo_prompt(example):
    """Format a problem as a chat prompt for GRPO."""
    problem = example["problem"]
    prompt_text = (
        "Solve this math problem step by step. "
        "Use Python code when calculations are needed.\n"
        "Put your final numerical answer in \\boxed{}.\n\n"
        f"Problem: {problem}"
    )
    answer = str(int(float(example["answer"]))) if example["answer"] is not None else "0"
    return {
        "prompt": [{"role": "user", "content": prompt_text}],
        "answer": answer,
    }

# Format all datasets
grpo_aime = ds_aime.map(make_grpo_prompt, remove_columns=ds_aime.column_names)
grpo_amc = ds_amc.map(make_grpo_prompt, remove_columns=ds_amc.column_names)

# Force both to have identical column types (datasets caches can cause type mismatches)
grpo_aime = grpo_aime.cast_column("answer", Value("string"))
grpo_amc = grpo_amc.cast_column("answer", Value("string"))

# Now safe to concatenate
grpo_dataset = concatenate_datasets([grpo_aime, grpo_amc])
print(f"GRPO training set: {len(grpo_dataset)} problems")
print(f"Each problem generates 8 rollouts -> {len(grpo_dataset) * 8} training samples/epoch")

---
## Part 3: GRPO (Group Relative Policy Optimization)

### Why RL after SFT?

SFT teaches the model to **copy** the format of solutions. But copying isn't reasoning.
A model might learn to write plausible-looking math that's actually wrong.

RL fixes this by giving **feedback on correctness**:

```
SFT:  "Here's how to write a solution"  (format learning)
RL:   "Was your answer RIGHT?"           (correctness learning)
```

### How GRPO works (simplified):

```
For each math problem:
  1. Generate G solutions (e.g., G=8)
  2. Check each: correct (+1) or wrong (0)
  3. Compute advantage = reward - group_mean_reward
     - If 3/8 solutions are correct, mean = 0.375
     - Correct solutions get advantage = 1.0 - 0.375 = +0.625
     - Wrong solutions get advantage = 0.0 - 0.375 = -0.375
  4. Update model: increase probability of high-advantage solutions,
     decrease probability of low-advantage solutions
```

### Why GRPO over PPO?

PPO (used in ChatGPT training) needs a separate "critic" model to estimate
how good each state is. GRPO skips this by comparing within the group.
This is simpler, uses less memory, and works great for math where
rewards are binary (right/wrong).

### The reward function

For math, the reward is beautifully simple:
- Extract the answer from `\boxed{...}`
- Compare to ground truth
- Reward = 1.0 if correct, 0.0 if wrong

No learned reward model needed! This is called a **verifiable reward** -
math has an objective right answer.

In [ ]:
# ============================================
# GRPO REWARD FUNCTION
# ============================================
# This is the core of RL for math: check if the answer is correct.

import re

def extract_boxed_answer(text):
    """Extract content from \\boxed{...} in model output."""
    # Find the LAST \boxed (model might write multiple)
    idx = text.rfind(r"\boxed")
    if idx < 0:
        return None
    
    # Match braces to find the complete \boxed{...}
    i, depth = idx, 0
    while i < len(text):
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
            if depth == 0:
                # Extract content between \boxed{ and }
                start = text.index("{", idx) + 1
                return text[start:i].strip()
        i += 1
    return None


def normalize_answer(text):
    """Clean up answer string for comparison."""
    if not text:
        return ""
    # Remove LaTeX formatting
    for remove in [r"\text{", "}", r"\mathrm{", "$", ",", " ", "%"]:
        text = text.replace(remove, "")
    return text.strip()


def correctness_reward(prompts, completions, answer, **kwargs):
    """
    GRPO reward function.
    
    TRL passes completions as chat message lists, e.g.:
      [[{"role": "assistant", "content": "The answer is \\boxed{42}"}], ...]
    
    Args:
        prompts: List of prompt message lists (unused here)
        completions: List of completion message lists
        answer: List of ground-truth answers (from dataset)
    
    Returns:
        List of rewards (1.0 = correct, 0.0 = wrong)
    """
    rewards = []
    for completion, true_answer in zip(completions, answer):
        # Extract the text content from the chat message list
        if isinstance(completion, list):
            text = completion[-1]["content"] if completion else ""
        elif isinstance(completion, dict):
            text = completion.get("content", "")
        else:
            text = str(completion)
        
        extracted = extract_boxed_answer(text)
        normalized = normalize_answer(extracted) if extracted else ""
        
        # Compare extracted answer to ground truth
        is_correct = (normalized == str(true_answer).strip())
        rewards.append(1.0 if is_correct else 0.0)
    
    return rewards


# Test the reward function (both formats)
test_text = r"The answer is \boxed{42}"
# As raw string
print(f"Extracted: {extract_boxed_answer(test_text)}")
# As chat message list (how TRL actually passes it)
test_chat = [[{"role": "assistant", "content": test_text}]]
print(f"Reward (correct): {correctness_reward([], test_chat, ['42'])}")
print(f"Reward (wrong):   {correctness_reward([], test_chat, ['43'])}")

In [ ]:
# ============================================
# PREPARE GRPO TRAINING DATA
# ============================================
# For GRPO we need problems with known answers.
# We use AIME + AMC validation sets (173 competition-level problems).
#
# WHY competition problems for RL?
# - Easy problems: model gets 100% right -> no learning signal
# - Impossible problems: model gets 0% right -> no learning signal  
# - Sweet spot: model gets 20-80% right -> RL can differentiate good/bad attempts

from datasets import load_dataset, concatenate_datasets

ds_aime = load_dataset("AI-MO/aimo-validation-aime", split="train")
ds_amc = load_dataset("AI-MO/aimo-validation-amc", split="train")

# Also use the NuminaMath-TIR test split
ds_tir_test = load_dataset("AI-MO/NuminaMath-TIR", split="test")

print(f"AIME: {len(ds_aime)} problems")
print(f"AMC: {len(ds_amc)} problems")
print(f"TIR test: {len(ds_tir_test)} problems")

In [ ]:
# Format problems as prompts for GRPO
# GRPOTrainer needs a 'prompt' column (list of messages) and any extra columns
# for the reward function (we pass 'answer').
from datasets import Features, Value, Sequence

def make_grpo_prompt(example):
    """Format a problem as a chat prompt for GRPO."""
    problem = example["problem"]
    prompt_text = (
        "Solve this math problem step by step. "
        "Use Python code when calculations are needed.\n"
        "Put your final numerical answer in \\boxed{}.\n\n"
        f"Problem: {problem}"
    )
    # IMPORTANT: cast answer to string so both datasets have matching type
    answer = str(int(float(example["answer"]))) if example["answer"] is not None else "0"
    return {
        "prompt": [{"role": "user", "content": prompt_text}],
        "answer": answer,
    }

# Format all datasets
grpo_aime = ds_aime.map(make_grpo_prompt, remove_columns=ds_aime.column_names)
grpo_amc = ds_amc.map(make_grpo_prompt, remove_columns=ds_amc.column_names)

# Force both to have identical column types (datasets caches can cause type mismatches)
grpo_aime = grpo_aime.cast_column("answer", Value("string"))
grpo_amc = grpo_amc.cast_column("answer", Value("string"))

# Combine into one GRPO dataset
grpo_dataset = concatenate_datasets([grpo_aime, grpo_amc])
print(f"GRPO training set: {len(grpo_dataset)} problems")
print(f"Each problem generates 8 rollouts -> {len(grpo_dataset) * 8} training samples/epoch")

In [ ]:
# Switch model back to training mode
FastLanguageModel.for_training(model)

from trl import GRPOConfig, GRPOTrainer  # Clean import - no mocking needed on Lambda!

# Configure GRPO
grpo_config = GRPOConfig(
    output_dir="./grpo_checkpoint",
    num_generations=8,              # G=8 solutions per problem
    max_completion_length=1024,     # Leave room for prompt within 2048 seq len
    max_prompt_length=512,          # Cap prompt length
    per_device_train_batch_size=2,  # 1×2×4=8, divisible by num_generations=8
    gradient_accumulation_steps=4,  # Effective batch for optimizer = 8 problems
    learning_rate=5e-6,             # Low LR for RL (10x lower than SFT)
    lr_scheduler_type="cosine",
    warmup_steps=50,
    num_train_epochs=3,
    bf16=True,
    logging_steps=1,
    save_steps=50,
    save_total_limit=2,
    use_vllm=False,                 # Use model.generate() (simpler, works with LoRA)
    seed=42,
)

grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    reward_funcs=[correctness_reward],
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
)

print(f"\nGRPO ready! {len(grpo_dataset)} problems × 3 epochs × 8 rollouts each")

In [ ]:
# ============================================
# RUN GRPO TRAINING (with auto-resume support)
# ============================================
# If training was interrupted, just re-run this cell.
# It automatically finds the last checkpoint and resumes from there.
#
# Watch these metrics:
#   - reward/mean: Should INCREASE (0.2-0.4 → 0.5-0.7)
#   - reward/std: Moderate is good (too low = collapsed, too high = erratic)

import os

# Auto-detect if there's a checkpoint to resume from
checkpoint_dir = "./grpo_checkpoint"
resume = None
if os.path.exists(checkpoint_dir):
    checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint-")]
    if checkpoints:
        latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
        resume = os.path.join(checkpoint_dir, latest)
        print(f"✓ Resuming from {resume}")
    else:
        print("Starting GRPO from scratch")
else:
    print("Starting GRPO from scratch")

grpo_stats = grpo_trainer.train(resume_from_checkpoint=resume)
print(f"\nGRPO training complete!")
print(f"Final training loss: {grpo_stats.training_loss:.4f}")

In [ ]:
# Save the final model (SFT + GRPO)
model.save_pretrained("./grpo_checkpoint")
tokenizer.save_pretrained("./grpo_checkpoint")
print("Final model saved!")

---
## Part 4: Evaluation

Let's test our model on a few problems to see the difference between:
- Base model (no training)
- After SFT only
- After SFT + GRPO

In [ ]:
# Quick evaluation on AIME problems
FastLanguageModel.for_inference(model)

eval_problems = ds_aime.select(range(min(10, len(ds_aime))))

correct = 0
total = 0

for example in eval_problems:
    problem = example["problem"]
    true_answer = str(example["answer"])
    
    messages = [{"role": "user", "content": (
        "Solve this math problem step by step. "
        "Use Python code when calculations are needed.\n"
        "Put your final numerical answer in \\boxed{}.\n\n"
        f"Problem: {problem}"
    )}]
    
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=2048,
            temperature=0.7,
            do_sample=True,
        )
    
    response = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    predicted = extract_boxed_answer(response)
    normalized = normalize_answer(predicted) if predicted else ""
    is_correct = (normalized == true_answer)
    correct += is_correct
    total += 1
    
    status = "OK" if is_correct else "WRONG"
    print(f"[{status}] Predicted={normalized}, True={true_answer}")

print(f"\nAccuracy: {correct}/{total} = {correct/total:.0%}")
print("\nNote: With SC-TIR (multiple samples + majority voting), accuracy will be MUCH higher!")
print("This single-sample eval is just a sanity check.")

---
## Part 5: Export Model for Kaggle Submission

To use your trained model in the competition submission:

1. **Upload to HuggingFace** -> attach as Kaggle Model input
2. Or **save as Kaggle Dataset** -> attach to submission notebook

We'll do option 1 (HuggingFace) since it's more flexible.

In [ ]:
# ============================================
# EXPORT: Merge LoRA + Upload to HuggingFace
# ============================================
# This merges the LoRA adapters back into the base model,
# creating a standalone model (~14GB) you can use anywhere.

# Step 1: Merge LoRA into base weights
merged_model = model.merge_and_unload()
merged_model.save_pretrained("./final_model")
tokenizer.save_pretrained("./final_model")
print("✓ Merged model saved to ./final_model")

# Step 2: Upload to HuggingFace
# First login: run `huggingface-cli login` in terminal, or set HF_TOKEN env var
from huggingface_hub import HfApi

REPO_ID = "YOUR_USERNAME/aimo3-qwen-math-7b-sft-grpo"  # <-- CHANGE THIS

api = HfApi()
api.create_repo(REPO_ID, exist_ok=True, repo_type="model")
api.upload_folder(
    folder_path="./final_model",
    repo_id=REPO_ID,
    repo_type="model",
)
print(f"✓ Uploaded to https://huggingface.co/{REPO_ID}")
print(f"\nNext: Go to Kaggle → Add Model Input → search for '{REPO_ID}' → attach to submission notebook")

In [ ]:
# Option B: Save as Kaggle Dataset output
# After this notebook runs, go to the Output tab and click "New Dataset"
# Then attach that dataset in your submission notebook.

import shutil
import os

kaggle_output = "/kaggle/working/final_model"
if os.path.exists("/kaggle/working"):
    shutil.copytree("./final_model", kaggle_output, dirs_exist_ok=True)
    print(f"Model saved to {kaggle_output}")
    print("After this notebook finishes, create a Dataset from the Output tab.")
else:
    print("Not running on Kaggle - skip this cell")

---
## Summary: What You Just Did

```
Qwen2.5-Math-7B-Instruct (base)
    |
    v  SFT on NuminaMath-TIR (72K problems)
    |  -> Model learns TIR format (code + reasoning + \boxed)
    |
    v  GRPO on AIME/AMC problems (173 problems x 8 rollouts)
    |  -> Model learns to get the RIGHT answer, not just plausible ones
    |
    v
YOUR TRAINED MODEL
    -> Use in submission notebook with SC-TIR (32 samples + majority vote)
```

### Key concepts learned:
- **SFT** = supervised learning from examples (format + style)
- **LoRA** = efficient training by only updating small adapter matrices
- **GRPO** = RL that compares solutions within a group (no critic model needed)
- **Binary rewards** = math is great for RL because answers are verifiable

### Next steps:
1. Run the submission notebook with your trained model
2. Try more GRPO epochs or harder problems
3. Try a larger base model (Qwen3.5-27B) if you have compute